## Summary: Improvements Made

### ✅ Issues Fixed

| Issue | Solution | Result |
|-------|----------|--------|
| **Overfitting** | Added Dropout, BatchNorm, L2 regularization | Train/Val accuracy closer |
| **Poor generalization** | Data augmentation (flip, brightness, contrast) | Better on unseen data |
| **Unbalanced splits** | Stratified train/val split | Fair class representation |
| **No early stopping** | EarlyStopping + ReduceLROnPlateau callbacks | Prevents unnecessary training |
| **High learning rate** | Reduced 1e-3 → 5e-4 | Stable convergence |

### 📊 Expected Results
- **Training Accuracy**: High (95%+)
- **Validation Accuracy**: Should now be much closer to training (~70-85%)
- **Loss Gap**: Significantly reduced

### 🚀 Next Steps for Deployment
1. Extract embeddings from test set
2. Create reference gallery from known identities
3. Use cosine similarity for face recognition
4. Fine-tune similarity threshold based on use case

In [ ]:
# 13. INFERENCE: FACE MATCHING AND RECOGNITION
# ===============================================

def get_embedding(embedding_model, img_path):
    """Extract embedding from an image."""
    img = cv2.imread(img_path)
    if img is None:
        raise ValueError(f"Cannot read image: {img_path}")
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
    img_batch = np.expand_dims(img_resized.astype('float32') / 255.0, axis=0)
    
    return embedding_model.predict(img_batch, verbose=0)[0]

def cosine_similarity(emb1, emb2):
    """Calculate cosine similarity between two embeddings."""
    return np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))

def find_matches(query_embedding, gallery_embeddings, gallery_labels, 
                 gallery_names, threshold=0.5, top_k=3):
    """
    Find best matches in gallery for a query embedding.
    
    Parameters:
    - threshold: minimum similarity score (0-1)
    - top_k: return top k matches
    """
    similarities = []
    for gallery_emb in gallery_embeddings:
        sim = cosine_similarity(query_embedding, gallery_emb)
        similarities.append(sim)
    
    similarities = np.array(similarities)
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        person_id = gallery_labels[idx]
        person_name = gallery_names[person_id]
        sim = similarities[idx]
        if sim >= threshold:
            results.append({
                'name': person_name,
                'similarity': float(sim),
                'match': True
            })
    
    if not results:
        results.append({
            'name': 'Unknown',
            'similarity': float(similarities[top_indices[0]]),
            'match': False
        })
    
    return results

# Example usage:
# query_img_path = '/path/to/query/image.jpg'
# query_emb = get_embedding(embedding_model, query_img_path)
# matches = find_matches(query_emb, gallery_embeddings, gallery_labels, 
#                       gallery_names, threshold=0.5, top_k=3)
# for match in matches:
#     print(f"{match['name']}: {match['similarity']:.4f} (Match: {match['match']})")

print("✓ Inference functions ready!")
print("  - get_embedding()")
print("  - cosine_similarity()")
print("  - find_matches()")


## 12. Inference: Face Matching and Recognition

In [ ]:
# 12. CREATE REFERENCE GALLERY FOR RECOGNITION
# ===============================================

def create_gallery(embedding_model, images_dir, output_path):
    """
    Create reference embeddings from a gallery of known people.
    
    Directory structure expected:
    images_dir/
        person_A/
            image1.jpg
            image2.jpg
        person_B/
            image1.jpg
            ...
    """
    embeddings_list = []
    labels_list = []
    label_to_name = {}
    
    person_dirs = sorted([d for d in os.listdir(images_dir) 
                         if os.path.isdir(os.path.join(images_dir, d))])
    
    print(f"Processing {len(person_dirs)} people...")
    
    for idx, person_folder in enumerate(person_dirs):
        person_path = os.path.join(images_dir, person_folder)
        label_to_name[idx] = person_folder
        
        img_files = [f for f in os.listdir(person_path)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
        for img_name in img_files:
            img_path = os.path.join(person_path, img_name)
            try:
                img = cv2.imread(img_path)
                if img is None:
                    continue
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
                img_batch = np.expand_dims(img_resized.astype('float32') / 255.0, axis=0)
                
                emb = embedding_model.predict(img_batch, verbose=0)[0]
                embeddings_list.append(emb)
                labels_list.append(idx)
            except Exception as e:
                print(f"  ⚠ Error processing {img_path}: {e}")
    
    embeddings_array = np.array(embeddings_list)
    labels_array = np.array(labels_list)
    
    # Save embeddings and metadata
    np.save(os.path.join(output_path, 'gallery_embeddings.npy'), embeddings_array)
    
    gallery_info = {
        'label_to_name': label_to_name,
        'labels': labels_array.tolist(),
        'num_embeddings': len(embeddings_array),
        'num_people': len(label_to_name)
    }
    with open(os.path.join(output_path, 'gallery_info.json'), 'w') as f:
        json.dump(gallery_info, f, indent=2)
    
    print(f"✓ Gallery created:")
    print(f"  - Total embeddings: {len(embeddings_array)}")
    print(f"  - Total people: {len(label_to_name)}")
    
    return embeddings_array, labels_array, label_to_name

# Example: Create gallery (uncomment and modify path as needed)
# gallery_dir = '/kaggle/input/your-reference-gallery'
# gallery_output = os.path.join(MODEL_DIR, 'gallery')
# os.makedirs(gallery_output, exist_ok=True)
# gallery_embeddings, gallery_labels, gallery_names = create_gallery(
#     embedding_model, gallery_dir, gallery_output
# )


## 11. Create Gallery for Inference

In [ ]:
# 11. SAVE MODELS
# ================

# Save full model with classifier head
model.save(os.path.join(MODEL_DIR, 'myFace_model_complete.h5'))
print(f"✓ Full model saved: myFace_model_complete.h5")

# Extract and save embedding-only model (for inference)
embedding_model = keras.Model(model.input, model.get_layer('l2_norm').output)
embedding_model.save(os.path.join(MODEL_DIR, 'myFace_embedding_model.h5'))
print(f"✓ Embedding model saved: myFace_embedding_model.h5")

# Save label mapping
label_mapping = {
    'id2label': id2label,
    'label2id': label2id,
    'num_classes': num_classes
}
with open(os.path.join(MODEL_DIR, 'label_mapping.json'), 'w') as f:
    json.dump(label_mapping, f, indent=2)
print(f"✓ Label mapping saved: label_mapping.json")

# Save model info
model_info = {
    'img_size': IMG_SIZE,
    'embedding_dim': 128,
    'num_classes': num_classes,
    'num_parameters': int(model.count_params()),
    'val_accuracy': float(val_accuracy),
    'val_loss': float(val_loss)
}
with open(os.path.join(MODEL_DIR, 'model_info.json'), 'w') as f:
    json.dump(model_info, f, indent=2)
print(f"✓ Model info saved: model_info.json")

print("\n✓ All models and metadata saved successfully!")


## 10. Model Saving and Export

In [ ]:
# 10. MODEL EVALUATION
# ====================

# Evaluate on validation set
val_loss, val_accuracy = model.evaluate(val_ds, verbose=0)
print(f"\n{'='*60}")
print(f"FINAL VALIDATION METRICS")
print(f"{'='*60}")
print(f"Validation Loss:     {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f} ({val_accuracy*100:.2f}%)")
print(f"{'='*60}\n")

# Get predictions on validation set
from sklearn.metrics import classification_report, confusion_matrix

y_pred_probs = model.predict(val_ds, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# True labels from val_ds
y_true = np.concatenate([y for _, y in val_ds], axis=0)

# Classification report
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=list(id2label.values())))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
print(f"\nConfusion Matrix shape: {cm.shape}")
print("(First 5 classes shown)")
print(cm[:5, :5])


## 9. Model Evaluation and Metrics

In [ ]:
# 9. VISUALIZE TRAINING HISTORY
# ==============================

def plot_training_history(history, title, phase):
    """Plot loss and accuracy curves."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Loss
    axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
    axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
    axes[0].set_title(f'{phase} - Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
    axes[1].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
    axes[1].set_title(f'{phase} - Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_DIR, f'training_history_{phase.lower().replace(" ", "_")}.png'), dpi=100)
    plt.show()

# Plot both phases
plot_training_history(history_phase1, 'Facial Recognition Training', 'Phase 1: Head Training')
plot_training_history(history_phase2, 'Facial Recognition Training', 'Phase 2: Fine-tuning')

print("✓ Training history plots saved")


## 8. Training History Visualization

In [ ]:
# 8. TRAINING PHASE 2: FINE-TUNING
# ==================================

# Unfreeze the last 30 layers of backbone for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

print(f"✓ Unfroze last 30 layers of backbone for fine-tuning")

# Recompile with lower learning rate
model.compile(
    optimizer=optimizers.Adam(learning_rate=LEARNING_RATE_FINE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ✅ Same callbacks for phase 2
callbacks_phase2 = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-8,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        os.path.join(MODEL_DIR, 'best_finetuned_model.h5'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=0
    )
]

print("=" * 60)
print("PHASE 2: Fine-tuning Backbone (Last 30 Layers)")
print("=" * 60)

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE,
    callbacks=callbacks_phase2,
    verbose=1
)

print("✓ Phase 2 training completed")


## 7. Training Phase 2: Fine-tuning the Backbone

In [ ]:
# 7. TRAINING PHASE 1: CLASSIFICATION HEAD
# ==========================================
# ✅ FIXES: Added Early Stopping and ReduceLROnPlateau callbacks

model.compile(
    optimizer=optimizers.Adam(learning_rate=LEARNING_RATE_HEAD),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ✅ Callbacks to prevent overfitting
callbacks_phase1 = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        os.path.join(MODEL_DIR, 'best_head_model.h5'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=0
    )
]

print("=" * 60)
print("PHASE 1: Training Classification Head")
print("=" * 60)

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks_phase1,
    verbose=1
)

print("✓ Phase 1 training completed")


## 6. Training Phase 1: Classification Head

In [ ]:
# 6. BUILD EMBEDDING MODEL WITH REGULARIZATION
# ==============================================
# ✅ FIXES: Added Dropout, L2 regularization, BatchNorm for better generalization

def build_embedding_model(input_shape=(160, 160, 3), num_classes=None):
    """
    Build MobileNetV2 + embedding layer + optional classifier head.
    
    Features:
    - Pre-trained MobileNetV2 backbone (frozen initially)
    - BatchNormalization + Dropout layers
    - L2 regularization on dense layers
    - 128-dim embedding with L2 normalization
    """
    # Load pre-trained MobileNetV2
    base_model = keras.applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet',
        pooling='avg'
    )
    base_model.trainable = False  # Freeze initially
    
    inputs = keras.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    
    # ✅ BatchNorm + Dropout for regularization
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    # ✅ Embedding layer with L2 regularization
    x = layers.Dense(
        128,
        activation=None,
        kernel_regularizer=keras.regularizers.l2(1e-4),
        name='embedding'
    )(x)
    x = layers.BatchNormalization()(x)
    
    # L2 normalization
    embeddings = layers.Lambda(
        lambda v: tf.math.l2_normalize(v, axis=1),
        name='l2_norm'
    )(x)
    
    if num_classes is not None:
        # Classification head
        x = layers.Dropout(0.4)(embeddings)
        classifier = layers.Dense(
            num_classes,
            activation='softmax',
            kernel_regularizer=keras.regularizers.l2(1e-4),
            name='classifier'
        )(x)
        model = keras.Model(inputs, classifier)
    else:
        model = keras.Model(inputs, embeddings)
    
    return model, base_model

# Build model
model, base_model = build_embedding_model(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    num_classes=num_classes
)

print(f"✓ Model built successfully")
print(f"  - Classes: {num_classes}")
print(f"  - Total parameters: {model.count_params():,}")


## 5. Model Architecture with Regularization

In [ ]:
# 5. DATA PIPELINE WITH AUGMENTATION
# ====================================
# ✅ FIXES: Missing data augmentation causing overfitting

def parse_image(img_path, label):
    """Parse image from file path."""
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

def augment_image(img, label):
    """
    Apply random augmentations to training images.
    ✅ Improves generalization and prevents memorization
    """
    # Random horizontal flip
    img = tf.image.random_flip_left_right(img)
    
    # Random vertical flip (subtle)
    if tf.random.uniform(()) > 0.7:
        img = tf.image.random_flip_up_down(img)
    
    # Random brightness adjustment
    img = tf.image.random_brightness(img, 0.15)
    
    # Random contrast adjustment
    img = tf.image.random_contrast(img, 0.8, 1.2)
    
    # Random saturation
    img = tf.image.random_saturation(img, 0.8, 1.2)
    
    # Clamp values to [0, 1]
    img = tf.clip_by_value(img, 0.0, 1.0)
    
    return img, label

def create_dataset(paths, labels, batch_size, training=False):
    """Create tf.data.Dataset with optimizations."""
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
    
    if training:
        # ✅ Larger shuffle buffer for better randomization
        ds = ds.shuffle(buffer_size=10000)
        # ✅ Apply augmentations
        ds = ds.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)
    
    # Batch and prefetch
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# Create datasets
train_ds = create_dataset(train_paths, train_labels, BATCH_SIZE, training=True)
val_ds = create_dataset(val_paths, val_labels, BATCH_SIZE, training=False)

print(f"✓ Datasets created successfully")
print(f"  - Train batches: {len(train_ds)}")
print(f"  - Val batches: {len(val_ds)}")


## 4. Data Pipeline with Augmentation

In [ ]:
# 4. LOAD PATHS AND LABELS
def load_image_paths_and_labels(txt_file):
    """Load image paths and labels from text file."""
    paths, labels = [], []
    label_to_id = {}
    
    with open(txt_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 2:
                continue
            img_path, label = parts[0], parts[1]
            if label not in label_to_id:
                label_to_id[label] = len(label_to_id)
            paths.append(img_path)
            labels.append(label_to_id[label])
    
    return np.array(paths), np.array(labels), label_to_id

# Load data
train_paths, train_labels, label2id = load_image_paths_and_labels(
    os.path.join(SPLIT_DIR, 'train.txt')
)
val_paths, val_labels, _ = load_image_paths_and_labels(
    os.path.join(SPLIT_DIR, 'val.txt')
)
num_classes = len(label2id)
id2label = {v: k for k, v in label2id.items()}

print(f"✓ Data loaded successfully")
print(f"  - Classes: {num_classes}")
print(f"  - Train samples: {len(train_labels)}")
print(f"  - Val samples: {len(val_labels)}")


In [ ]:
# 3. DATA PREPARATION WITH STRATIFIED SPLIT
# ============================================

def prepare_stratified_splits(data_dir, output_dir, val_split=0.2):
    """
    Create train/val splits with stratification to ensure balanced class representation.
    ✅ Fixes: non-stratified split causing unbalanced datasets
    """
    all_images = []
    all_labels = []

    # Collect all images and their labels
    for person in sorted(os.listdir(data_dir)):
        person_dir = os.path.join(data_dir, person)
        if not os.path.isdir(person_dir):
            continue
        
        images = [os.path.join(person, f) for f in os.listdir(person_dir)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
        if len(images) < 2:
            print(f"⚠ Skipping {person}: less than 2 images")
            continue
        
        for img in images:
            all_images.append(img)
            all_labels.append(person)
    
    print(f"Total images collected: {len(all_images)} from {len(set(all_labels))} classes")
    
    # ✅ STRATIFIED SPLIT: ensures each class is represented proportionally
    train_imgs, val_imgs, _, _ = train_test_split(
        all_images, all_labels,
        test_size=val_split,
        random_state=42,
        stratify=all_labels  # ✅ KEY: stratify by class
    )
    
    # Save to files
    with open(os.path.join(output_dir, 'train.txt'), 'w') as f:
        for img in train_imgs:
            label = img.split(os.sep)[0]  # First folder = person name
            f.write(f"{os.path.join(data_dir, img)} {label}\n")
    
    with open(os.path.join(output_dir, 'val.txt'), 'w') as f:
        for img in val_imgs:
            label = img.split(os.sep)[0]
            f.write(f"{os.path.join(data_dir, img)} {label}\n")
    
    print(f"✓ Train split: {len(train_imgs)} images")
    print(f"✓ Val split: {len(val_imgs)} images")
    return len(train_imgs), len(val_imgs)

# Run preparation if needed
# train_count, val_count = prepare_stratified_splits(DATA_DIR, SPLIT_DIR)


## 3. Data Preparation with Stratified Split

In [ ]:
# 2. CONFIGURATION AND PARAMETERS
# ============================================

# Paths configuration
DATA_DIR = '/kaggle/input/your-dataset'  # Change to your Kaggle dataset
SPLIT_DIR = '/kaggle/working/splits'
MODEL_DIR = '/kaggle/working/models'
os.makedirs(SPLIT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Model parameters
IMG_SIZE = 160
BATCH_SIZE = 32
EPOCHS_HEAD = 15
EPOCHS_FINE = 30
LEARNING_RATE_HEAD = 5e-4  # ✅ Reduced from 1e-3
LEARNING_RATE_FINE = 5e-6   # ✅ Even lower for fine-tuning

print(f"✓ Configuration ready")
print(f"  - Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Epochs (head): {EPOCHS_HEAD}, (fine-tuning): {EPOCHS_FINE}")


## 2. Configuration and Data Setup

In [ ]:
# 1. IMPORT REQUIRED LIBRARIES
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import seaborn as sns

# Configure matplotlib for notebook display
%matplotlib inline
sns.set_style("darkgrid")

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU'))} GPU(s)")


# Facial Recognition Model Training

## Objective
Train a deep learning model for facial recognition using MobileNetV2 with embedding extraction and classification head. This notebook includes:
- Data augmentation for better generalization
- Regularization techniques (Dropout, L2 regularization)
- Early stopping and learning rate reduction
- Two-phase training: classification head + fine-tuning